1. Normalized + windowed time-series dataset
1.1. Conceptual setup (for a paper)

You have, for each experiment 
e∈{1,…,E}:

Physical model output:

𝑇𝑐(𝑒)(𝑡𝑖)𝑖=1,…,𝑛

Measured spindle current:

𝐼𝑞(𝑒)(𝑡𝑖)Iq(e)(ti)

You want to train a neural map 
𝑁𝜃:𝑥𝑖:𝑖+𝐿−1↦𝑦𝑖:𝑖+𝐿−1

↦yi:i+L−1
where
𝑥𝑖:𝑖+𝐿−1x
is a window of modeled torque (and possibly other features),
𝑦𝑖:𝑖+𝐿−1yi:i+L−1
	​
is the corresponding window of measured current,

L is the sequence length (“window size”).
This respects temporal structure, which a simple i.i.d. regression would ignore.
1.2. Normalization

For training stability, you usually normalize features to zero mean and unit variance:

𝑇~𝑐=𝑇𝑐−𝜇𝑇
𝐼~𝑞=𝐼𝑞−𝜇𝐼𝜎𝐼

Crucially, you estimate 

 on the training set only, then apply them to validation/test data.

In [9]:
def interpolate_nan(x):
    """Interpolate NaN values in a 1D array."""
    x = x.copy()
    nans = np.isnan(x)
    
    if np.all(nans):
        raise ValueError("Array contains only NaNs — cannot interpolate.")
    
    x[nans] = np.interp(
        np.flatnonzero(nans),
        np.flatnonzero(~nans),
        x[~nans]
    )
    return x


In [ ]:
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# --- Load all measurement files ---
tool_type = 'C1030'

base = (f"../../../data/preprocessed/Mill/measurement_*_training_data.csv")
files = sorted(glob.glob(base))

all_Tc = []
all_Iq = []

for f in files:
    df = pd.read_csv(f)
    Tc = df["Tc_model"].values.astype(float)
    Iq = df["Iq_real"].values.astype(float)
    Tc_clean = interpolate_nan(Tc)
    Iq_clean = interpolate_nan(Iq)

    all_Tc.append(Tc_clean)
    all_Iq.append(Iq_clean)
## interpolate NaNs
Tc_clean = interpolate_nan(Tc)
Iq_clean = interpolate_nan(Iq)


# Concatenate all experiments (we keep separate arrays too if needed)
Tc_all = np.concatenate(all_Tc)
Iq_all = np.concatenate(all_Iq)

# --- Compute normalization on entire dataset (or better: just on training split later) ---
Tc_mean, Tc_std = Tc_all.mean(), Tc_all.std()
Iq_mean, Iq_std = Iq_all.mean(), Iq_all.std()

def normalize_Tc(x):
    return (x - Tc_mean) / Tc_std

def normalize_Iq(y):
    return (y - Iq_mean) / Iq_std

# --- Build windowed dataset ---
def build_windows(Tc_list, Iq_list, window_size=128, stride=1):
    X, Y = [], []
    for Tc, Iq in zip(Tc_list, Iq_list):
        Tc_norm = normalize_Tc(Tc)
        Iq_norm = normalize_Iq(Iq)
        n = len(Tc)
        for start in range(0, n - window_size + 1, stride):
            end = start + window_size
            X.append(Tc_norm[start:end])   # (L,)
            Y.append(Iq_norm[start:end])   # (L,)
    X = np.array(X)[..., None]  # shape: (num_windows, L, 1)
    Y = np.array(Y)[..., None]  # shape: (num_windows, L, 1)
    return X, Y

X, Y = build_windows(all_Tc, all_Iq, window_size=128, stride=16)

# Train/validation split
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, shuffle=True, random_state=42)


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (72,) + inhomogeneous part.

In [20]:
## check if Tc and Iq have NaNs
assert not np.isnan(X_train).any(), "NaNs found in X_train"
print(f'NaNs found in Y_train {np.isnan(Y_train).sum()}')
Y_train

NaNs found in Y_train 3442176


array([[[nan],
        [nan],
        [nan],
        ...,
        [nan],
        [nan],
        [nan]],

       [[nan],
        [nan],
        [nan],
        ...,
        [nan],
        [nan],
        [nan]],

       [[nan],
        [nan],
        [nan],
        ...,
        [nan],
        [nan],
        [nan]],

       ...,

       [[nan],
        [nan],
        [nan],
        ...,
        [nan],
        [nan],
        [nan]],

       [[nan],
        [nan],
        [nan],
        ...,
        [nan],
        [nan],
        [nan]],

       [[nan],
        [nan],
        [nan],
        ...,
        [nan],
        [nan],
        [nan]]])

## Keras baseline model

In [12]:
import tensorflow as tf
from tensorflow.keras import layers, models

timesteps = X_train.shape[1]
features = X_train.shape[2]   # 1 in our case

def build_baseline_conv_model(timesteps, features):
    inputs = layers.Input(shape=(timesteps, features))
    
    x = layers.Conv1D(filters=32, kernel_size=5, padding="same", activation="relu")(inputs)
    x = layers.Conv1D(filters=32, kernel_size=5, padding="same", activation="relu")(x)
    x = layers.Conv1D(filters=16, kernel_size=5, padding="same", activation="relu")(x)
    
    # Final layer outputs one channel (Iq prediction)
    outputs = layers.Conv1D(filters=1, kernel_size=1, padding="same", activation="linear")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model

baseline_model = build_baseline_conv_model(timesteps, features)
baseline_model.summary()

history = baseline_model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=30,
    batch_size=64
)


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 128, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 128, 32)        │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 128, 32)        │         5,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 128, 16)        │         2,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 128, 1)         │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,937 (31.00 KB)

 Trainable params: 7,937 (31.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
421/421 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 2/30
421/421 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 3/30
421/421 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 4/30
421/421 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 5/30
421/421 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 6/30
421/421 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 7/30
421/421 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 8/30
421/421 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 9/30
421/421 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 10/30
421/421 ━━━━━━━━━━━━━━━━━

In [8]:
print(np.isnan(X_train).any(), np.isnan(Y_train).any())

False True


## Keras LSTM model

In [5]:
def build_lstm_seq2seq_model(timesteps, features, hidden_units=64):
    inputs = layers.Input(shape=(timesteps, features))

    x = layers.LSTM(hidden_units, return_sequences=True)(inputs)
    x = layers.LSTM(hidden_units, return_sequences=True)(x)
    
    # Time-distributed dense over each timestep
    outputs = layers.TimeDistributed(layers.Dense(1))(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model

lstm_model = build_lstm_seq2seq_model(timesteps, features)
lstm_model.summary()

history_lstm = lstm_model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=40,
    batch_size=32
)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128, 64)        │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128, 64)        │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 128, 1)         │            65 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,985 (195.25 KB)

 Trainable params: 49,985 (195.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/40
841/841 ━━━━━━━━━━━━━━━━━━━━ 35s 36ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 2/40
841/841 ━━━━━━━━━━━━━━━━━━━━ 28s 34ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 3/40
841/841 ━━━━━━━━━━━━━━━━━━━━ 28s 33ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 4/40
841/841 ━━━━━━━━━━━━━━━━━━━━ 27s 32ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 5/40
841/841 ━━━━━━━━━━━━━━━━━━━━ 27s 32ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 6/40
841/841 ━━━━━━━━━━━━━━━━━━━━ 24s 29ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 7/40
841/841 ━━━━━━━━━━━━━━━━━━━━ 26s 31ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 8/40
841/841 ━━━━━━━━━━━━━━━━━━━━ 30s 35ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 9/40
841/841 ━━━━━━━━━━━━━━━━━━━━ 28s 33ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 10/40
245/841 ━━━━━━━━

KeyboardInterrupt: 